In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression


In [9]:
DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / "consumption_temp.csv")
print(df.info())
print(df['location'].value_counts())
df = df.sort_values(['time','location'])
df.head(5)
print(df.iloc[0:3,:])
print(df.iloc[-1,:])

<class 'pandas.DataFrame'>
RangeIndex: 49494 entries, 0 to 49493
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   time         49494 non-null  str    
 1   location     49494 non-null  str    
 2   consumption  49494 non-null  float64
 3   temperature  49494 non-null  float64
dtypes: float64(2), str(2)
memory usage: 2.8 MB
None
location
bergen         8641
oslo           8641
stavanger      8641
tromsø         8641
trondheim      8641
helsingfors    6289
Name: count, dtype: int64
                  time   location  consumption  temperature
0  2022-04-07 21:00:00     bergen     1.113325         -0.3
1  2022-04-07 21:00:00       oslo     4.092830          1.0
2  2022-04-07 21:00:00  stavanger     2.057858          1.3
time           2023-04-02 21:00:00
location                 trondheim
consumption               3.749047
temperature                   -3.8
Name: 49493, dtype: object


Ingen null-verdier, men ser det er mindre data for Helsinki

In [110]:
#Feature engineering
df['time'] = pd.to_datetime(df['time'])
#df["time"] = df["time"].dt.tz_localize("UTC").dt.tz_convert("Europe/Oslo")

df["date"] = df["time"].dt.normalize()
df["hour"] = df["time"].dt.hour

g = df.groupby("location")["consumption"]

for days in [6, 7, 14]:
    df[f"cons_lag{days}d"] = g.shift(24 * days)

# Daily level from the last fully known day (target date - 6 days)
daily = (df.groupby(["location", "date"])["consumption"]
           .agg(["mean", "max", "min"])
           .add_prefix("day6_").reset_index())
daily["date"] = daily["date"] + pd.Timedelta(days=6)
df = df.merge(daily, on=["location", "date"], how="left")

df["weekday"] = df["time"].dt.weekday

#Temporal split på treningsdata
cutoff = pd.to_datetime("2023-02-01")
train = df[df["date"] < cutoff - pd.Timedelta(days=6)]
print(train.shape[0]/df.shape[0])
test  = df[df["date"] >= cutoff]

X_train = train.drop(columns=['consumption'])
X_test = test.drop(columns=['consumption'])
y_train = train['consumption']
y_test = test['consumption']

0.8053097345132744


In [111]:
df.groupby("location").corrwith(df["consumption"])

C:\Users\truls\AppData\Local\Temp\ipykernel_18252\818135802.py:1: Pandas4Warning: DataFrameGroupBy.corrwith is deprecated
  df.groupby("location").corrwith(df["consumption"])


,time,consumption,temperature,date,hour,cons_lag6d,cons_lag7d,cons_lag14d,day6_mean,day6_max,day6_min,weekday
location,,,,,,,,,,,,
bergen,0.878427,1.0,-0.781353,0.878089,0.121455,0.957144,0.955804,0.945349,0.943199,0.946625,0.932147,0.035881
helsingfors,0.579756,1.0,-0.567863,0.579692,0.016677,0.886292,0.882710,0.824951,0.905770,0.892952,0.868408,-0.048967
oslo,0.874840,1.0,-0.861854,0.874586,0.091464,0.961329,0.959300,0.945112,0.952331,0.956009,0.944092,0.023422
stavanger,0.869575,1.0,-0.781747,0.869301,0.098375,0.962099,0.960282,0.946032,0.952664,0.956752,0.944023,0.027109
tromsø,0.828474,1.0,-0.863436,0.828084,0.140205,0.947524,0.948073,0.930925,0.929161,0.932898,0.918964,0.026939
trondheim,0.859147,1.0,-0.837779,0.858767,0.136859,0.941915,0.940876,0.934874,0.924384,0.928238,0.911910,0.042784


In [113]:
feats = ["cons_lag6d", "cons_lag7d", "cons_lag14d", "temperature", "hour", "weekday"]

train_lr = train.dropna(subset=feats)          # drop the rows from the whole table
test_lr  = test.dropna(subset=feats)

Xtr = pd.get_dummies(train_lr[feats + ["location"]], columns=["location", "hour", "weekday"])
Xte = pd.get_dummies(test_lr[feats + ["location"]],  columns=["location", "hour", "weekday"])
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

ytr = train_lr["consumption"]
yte = test_lr["consumption"]

lin_reg = LinearRegression()
lin_reg.fit(Xtr, ytr)

pred = lin_reg.predict(Xte)

print(f"linear regression error: {mean_absolute_error(pred, yte)}")

linear regression error: 0.46936546731158957


In [ ]:
print("Naive 7d:", mean_absolute_error(y_test, X_test["cons_lag7d"]))

feats = ["location", "hour", "weekday", "temperature",
         "cons_lag6d", "cons_lag7d", "cons_lag14d",
         "day6_mean", "day6_max", "day6_min"]
for d in (train, test):
    d["location"] = d["location"].astype("category")

model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                         enable_categorical=True, tree_method="hist", random_state=42)
model.fit(train[feats], train["consumption"])
pred = model.predict(test[feats])
print("XGB:", mean_absolute_error(test["consumption"], pred))

test = test.assign(pred=pred, err=lambda d: (d["consumption"] - d["pred"]).abs())
test.groupby("location")["err"].mean()




Naive 7d: 0.4583872547195623
XGB: 0.3768329358376043


location
bergen         0.233657
helsingfors    0.147940
oslo           0.974853
stavanger      0.557983
tromsø         0.106159
trondheim      0.240406
Name: err, dtype: float64

Er det bedre med jevnere feilfordeling, hvis kostnad ved imbalanse er eksponentiell?